# Kaggle Notebook: AI Detection at Scale — Binoculars + Benchmarks

This notebook runs on Kaggle's free GPU (T4 x2). It:
1. Clones the `ai-detection-at-scale` repo
2. Downloads data/models from GitHub Releases
3. Runs the stylometric detector on low-FPR and humanization tests
4. Runs the Binoculars baseline (Falcon-7B or GPT-2)
5. Saves results as Kaggle output files

**Recommended Kaggle settings:** GPU T4 x2, 12h runtime.

## 1. Clone repo and install dependencies

In [ ]:
# Turn on GPU first: Settings → Accelerator → GPU T4 x2
!git clone https://github.com/vedangvatsa/ai-detection-at-scale.git
%cd ai-detection-at-scale
!pip install -q -r requirements.txt
!pip install -q torch transformers uvicorn fastapi

## 2. Download data/models from GitHub Releases

In [ ]:
!python scripts/download_assets.py

## 3. Verify stylometric detectors still work

In [ ]:
import pandas as pd
df = pd.read_csv('results/extended_feature_comparison.csv')
print(df.to_string(index=False))

## 4. Run TPR at low FPR (if not already done)

In [ ]:
!python scripts/10_tpr_at_low_fpr.py

## 5. Run Binoculars baseline

Use `gpt2` for CPU/low-VRAM test. Use `falcon-7b` for the real comparison.
Kaggle T4 has 16GB VRAM — Falcon-7B in FP16 should fit.

In [ ]:
# Choose model: gpt2 (fast, small) or tiiuae/falcon-7b (real Binoculars comparison)
MODEL = "gpt2"
!python scripts/13_binoculars_baseline.py \
    --observer-model {MODEL} \
    --performer-model {MODEL} \
    --max-texts 2000

## 6. Run multi-signal ensemble (includes Binoculars if available)

In [ ]:
!python scripts/14_multi_signal_ensemble.py

## 7. View and save results

In [ ]:
import os, glob
for f in sorted(glob.glob('results/*.csv')):
    print(f"\n=== {f} ===")
    print(pd.read_csv(f).to_string(index=False))

# Save all results to /kaggle/output for download
os.makedirs('/kaggle/output', exist_ok=True)
for f in glob.glob('results/*.csv') + glob.glob('models/*.joblib'):
    !cp {f} /kaggle/output/